# Neural Network with Torch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, n_head, dropout=0.1):
        super().__init__()
        assert d_model % n_head == 0

        self.d_model = d_model
        self.n_head = n_head
        self.head_dim = d_model // n_head

        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # x: [batch, seq_len, d_model]
        # mask: [batch, seq_len, seq_len] or broadcastable, optional
        batch, seq_len, _ = x.shape

        # 1. Linear projection and dehead
        q = self.wq(x).view(batch, seq_len, self.n_head, self.head_dim).transpose(1, 2) # [b, n_head, seq, head_dim]
        k = self.wk(x).view(batch, seq_len, self.n_head, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(batch, seq_len, self.n_head, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        if mask